In [6]:
# -*- coding: utf-8 -*-
"""
Code to train and evaluate TF-IDF+LogReg and Transformer models
for overall sentiment classification based on star ratings.
"""
# 1. Install and import NLTK
!pip install nltk
import nltk
# Import necessary libraries
import pandas as pd
import numpy as np
import json
import re
import time
import os
from datetime import datetime

# Text Preprocessing
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer

# Scikit-learn for baseline models and metrics
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay
)

# PyTorch and Hugging Face Transformers
import torch
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    get_linear_schedule_with_warmup,
    DistilBertTokenizer, # Using DistilBERT as in the original notebook
    DistilBertForSequenceClassification
)

# Plotting (Optional, for confusion matrix)
import matplotlib.pyplot as plt
import seaborn as sns

import gc

# 2. Modified download function
def download_nltk_data():
    """Downloads necessary NLTK data packages."""
    packages = ['punkt', 'stopwords', 'wordnet', 'punkt_tab'] # Added 'punkt_tab' to the list
    for package in packages:
        try:
            nltk.data.find(f'corpora/{package}' if package != 'punkt' else f'tokenizers/{package}')
        except LookupError:
            print(f"Downloading NLTK package: {package}...")
            nltk.download(package, quiet=True)
    print("NLTK data check complete.")

# 3. Call the download function
download_nltk_data()

# --- Configuration & Setup ---

# Set random seed for reproducibility
SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# Check for GPU
device = torch.device('cuda') if torch.cuda.is_available() else torch.device('cpu')
print(f"Using device: {device}")

# --- Download NLTK data (if needed) ---
def download_nltk_data():
    """Downloads necessary NLTK data packages."""
    packages = ['punkt', 'stopwords', 'wordnet']
    for package in packages:
        try:
            nltk.data.find(f'corpora/{package}' if package != 'punkt' else f'tokenizers/{package}')
        except LookupError:
            print(f"Downloading NLTK package: {package}...")
            nltk.download(package, quiet=True)
    print("NLTK data check complete.")

download_nltk_data()

# --- File Paths & Data Loading ---
# Adjust paths based on whether running in Colab or locally
try:
    from google.colab import drive
    drive.mount('/content/drive')
    # Adjust Google Drive paths as needed
    review_file_path = '/content/drive/My Drive/Colab Notebooks/SentimentModels_Stars/yelp_academic_dataset_review.json'
    business_file_path = '/content/drive/My Drive/Colab Notebooks/SentimentModels_Stars/yelp_academic_dataset_business.json'
    # Define output directory for models if needed
    output_dir = '/content/drive/My Drive/Colab Notebooks/SentimentModels_Stars'
    print("Running in Colab environment.")
except ModuleNotFoundError:
    # Adjust local paths as needed
    review_file_path = r"G:\My Drive\CMPS 6730 NLP\FinalProject\yelp_academic_dataset_review.json"
    business_file_path = r"G:\My Drive\CMPS 6730 NLP\FinalProject\yelp_academic_dataset_business.json"
    # Define output directory for models if needed
    output_dir = './SentimentModels_Stars'
    print("Running in local environment.")

# Create output directory if it doesn't exist
os.makedirs(output_dir, exist_ok=True)
print(f"Model output directory: {output_dir}")

def load_data(review_path, business_path):
    """Loads review and business data, filters, and merges."""
    print(f"\nLoading review data from: {review_path}")
    start_time = time.time()
    try:
        # Load reviews in chunks for memory efficiency
        all_data = []
        chunksize = 100000
        for chunk in pd.read_json(review_path, lines=True, chunksize=chunksize):
            all_data.append(chunk[['review_id', 'business_id', 'stars', 'text', 'date']]) # Select columns early
        df_reviews = pd.concat(all_data, ignore_index=True)
    except Exception as e:
        print(f"Error loading review data: {e}")
        return None, None
    loading_time = time.time() - start_time
    print(f"Review data loaded in {loading_time:.2f} seconds. Shape: {df_reviews.shape}")

    print(f"\nLoading business data from: {business_path}")
    start_time = time.time()
    try:
        # Handle potential JSON errors in business data
        business_data = []
        with open(business_path, 'r', encoding='utf-8') as f:
            for line in f:
                try:
                    business_data.append(json.loads(line))
                except json.JSONDecodeError:
                    print(f"Skipping malformed line in businesses")
                    continue
        df_business = pd.DataFrame(business_data)
        df_business = df_business[['business_id', 'categories']] # Select only needed columns
    except Exception as e:
        print(f"Error loading business data: {e}")
        return df_reviews, None # Return reviews even if businesses fail
    loading_time = time.time() - start_time
    print(f"Business data loaded in {loading_time:.2f} seconds. Shape: {df_business.shape}")

    # Convert review date
    df_reviews['date'] = pd.to_datetime(df_reviews['date'])

    # Filter businesses for Restaurants/Food
    df_business['categories'] = df_business['categories'].fillna('')
    target_restaurants = df_business[
        df_business['categories'].str.contains('Restaurant|Food', case=False, regex=True)
    ].copy()
    target_business_ids = set(target_restaurants['business_id'])
    print(f"\nFound {len(target_business_ids)} unique target restaurant business IDs.")

    if not target_business_ids:
        print("Warning: No restaurant business IDs found. Check categories.")
        # Decide how to proceed - here we continue with only date filtering
        # return df_reviews, None

    # Filter reviews by date (e.g., 2019-2024)
    start_date = datetime(2019, 1, 1)
    end_date = datetime(2024, 12, 31)
    df_reviews_filtered_date = df_reviews[
        (df_reviews['date'] >= start_date) & (df_reviews['date'] <= end_date)
    ].copy()
    print(f"Reviews filtered by date ({start_date.date()} to {end_date.date()}). Shape: {df_reviews_filtered_date.shape}")

    # Filter reviews by target business IDs
    if target_business_ids:
        df_filtered = df_reviews_filtered_date[
            df_reviews_filtered_date['business_id'].isin(target_business_ids)
        ].copy()
        print(f"Reviews filtered by target business IDs. Shape: {df_filtered.shape}")
    else:
        print("Skipping business ID filtering as no target IDs were found.")
        df_filtered = df_reviews_filtered_date # Use date-filtered if no business match

    if df_filtered.empty:
        print("\nWARNING: No reviews match the filtering criteria. Cannot proceed.")
        return None, None

    print(f"\nFinal shape for processing: {df_filtered.shape}")
    # Select final columns needed
    df_processed = df_filtered[['review_id', 'text', 'stars']].copy()
    print(df_processed.head())

    # Optional: Cleanup memory
    del df_reviews, df_business, df_reviews_filtered_date, target_restaurants, df_filtered
    import gc
    gc.collect()
    print("\nIntermediate dataframes cleaned up.")

    return df_processed

# Load the data
# Load the data
df_processed = load_data(review_file_path, business_file_path)  # Get the single returned DataFrame
if df_processed is None or df_processed.empty:
    print("Exiting due to data loading/filtering issues.")
    # exit() # Uncomment to stop execution if needed

# --- Text Preprocessing Function ---
lemmatizer = WordNetLemmatizer()
stop_words_set = set(stopwords.words('english'))

def preprocess_text(text):
    """Cleans and preprocesses text data."""
    if not isinstance(text, str):
        return ""
    text = text.lower() # Lowercase
    text = re.sub(r'http\S+|www\S+|https\S+', '', text, flags=re.MULTILINE) # Remove URLs
    text = re.sub(r'\@\w+|\#', '', text) # Remove mentions and hashtags
    text = re.sub(r'[^\w\s]', '', text) # Remove punctuation
    text = re.sub(r'\d+', '', text) # Remove numbers
    tokens = word_tokenize(text) # Tokenize
    processed_tokens = [
        lemmatizer.lemmatize(word) for word in tokens
        if word not in stop_words_set and word.isalpha() and len(word) > 1 # Keep words > 1 char
    ]
    return ' '.join(processed_tokens)

# Apply preprocessing
if df_processed is not None and not df_processed.empty:
    print("\nStarting text preprocessing...")
    start_time = time.time()
    # Ensure 'text' column is string type, fill NaNs with empty string
    df_processed['text'] = df_processed['text'].astype(str).fillna('')
    df_processed['processed_text'] = df_processed['text'].apply(preprocess_text)
    processing_time = time.time() - start_time
    print(f"Text preprocessing completed in {processing_time:.2f} seconds.")

    # Display some examples
    print("\nOriginal vs Processed Text Examples:")
    for i in range(min(3, len(df_processed))):
        print(f"--- Example {i+1} ---")
        print("Original:", df_processed['text'].iloc[i][:200] + "...")
        print("Processed:", df_processed['processed_text'].iloc[i])
        print("-" * 20)

    # Drop rows where processed text became empty
    original_len = len(df_processed)
    df_processed.dropna(subset=['processed_text'], inplace=True)
    df_processed = df_processed[df_processed['processed_text'] != '']
    print(f"\nRemoved {original_len - len(df_processed)} rows with empty processed text.")
    print(f"Shape after cleaning empty processed texts: {df_processed.shape}")
else:
    print("Skipping preprocessing as df_processed is not available.")


# --- Create Target Variable from Stars ---
def map_stars_to_sentiment(stars):
    """Maps star rating to sentiment label: 0 (neg), 1 (neu), 2 (pos)."""
    if stars in [1, 2]:
        return 0 # Negative
    elif stars == 3:
        return 1 # Neutral
    elif stars in [4, 5]:
        return 2 # Positive
    else:
        return -1 # Should not happen with Yelp data, but good practice

if df_processed is not None and not df_processed.empty:
    print("\nMapping star ratings to sentiment labels...")
    df_processed['overall_sentiment_stars'] = df_processed['stars'].apply(map_stars_to_sentiment)
    # Verify mapping
    print("Value counts for 'overall_sentiment_stars':")
    print(df_processed['overall_sentiment_stars'].value_counts())
    # Drop any potential -1 mappings if they occurred
    df_processed = df_processed[df_processed['overall_sentiment_stars'] != -1]
    print(f"Shape after mapping stars: {df_processed.shape}")
else:
    print("Skipping star mapping as df_processed is not available.")


# --- Prepare Data for Models ---
if df_processed is not None and not df_processed.empty:
    X = df_processed['processed_text']
    Y = df_processed['overall_sentiment_stars']

    # Split data - Stratify by the new star-based sentiment
    print("\nSplitting data into training and testing sets...")
    X_train, X_test, Y_train, Y_test = train_test_split(
        X, Y,
        test_size=0.2,      # 80% train, 20% test
        random_state=SEED,
        stratify=Y         # Stratify based on the target variable
    )
    print(f"Training set size: {len(X_train)}")
    print(f"Testing set size: {len(X_test)}")
    print("\nTraining set label distribution:")
    print(Y_train.value_counts(normalize=True))
    print("\nTesting set label distribution:")
    print(Y_test.value_counts(normalize=True))

else:
    print("Cannot prepare data as df_processed is not available.")
    # exit() # Stop if data isn't ready

# --- Model 1: TF-IDF + Logistic Regression ---

if 'X_train' in locals(): # Check if data split was successful
    print("\n--- Training Model 1: TF-IDF + Logistic Regression ---")

    # 1. TF-IDF Vectorization
    print("Applying TF-IDF Vectorizer...")
    start_time = time.time()
    vectorizer = TfidfVectorizer(max_features=10000, ngram_range=(1, 2)) # Increased features, added bigrams
    X_train_tfidf = vectorizer.fit_transform(X_train)
    X_test_tfidf = vectorizer.transform(X_test)
    print(f"TF-IDF fitting/transformation done in {time.time() - start_time:.2f} seconds.")
    print(f"TF-IDF Matrix Shape (Train): {X_train_tfidf.shape}")

    # 2. Logistic Regression Training
    print("Training Logistic Regression model...")
    start_time = time.time()
    # Increased max_iter, consider trying 'saga' solver for large datasets if 'liblinear' is slow
    log_reg_model = LogisticRegression(
        solver='liblinear', # Good for high-dimensional sparse data
        random_state=SEED,
        max_iter=1000,     # Increase max iterations
        C=1.0              # Regularization strength (default)
    )
    log_reg_model.fit(X_train_tfidf, Y_train)
    print(f"Logistic Regression training done in {time.time() - start_time:.2f} seconds.")

    # 3. Evaluation
    print("\nEvaluating Logistic Regression model...")
    Y_pred_log_reg = log_reg_model.predict(X_test_tfidf)

    print("\nLogistic Regression Classification Report:")
    target_names = ['Negative (1-2 Stars)', 'Neutral (3 Stars)', 'Positive (4-5 Stars)']
    print(classification_report(Y_test, Y_pred_log_reg, target_names=target_names, digits=4))

    # Optional: Confusion Matrix Plot
    print("Generating Confusion Matrix for Logistic Regression...")
    try:
        cm_log_reg = confusion_matrix(Y_test, Y_pred_log_reg)
        disp_log_reg = ConfusionMatrixDisplay(confusion_matrix=cm_log_reg, display_labels=target_names)
        fig, ax = plt.subplots(figsize=(8, 6))
        disp_log_reg.plot(cmap=plt.cm.Blues, ax=ax, xticks_rotation='vertical')
        plt.title('Logistic Regression Confusion Matrix (Stars)')
        plt.tight_layout()
        plt.savefig(os.path.join(output_dir, 'log_reg_confusion_matrix_stars.png'))
        # plt.show() # Uncomment to display plot directly
        plt.close(fig) # Close the figure to prevent displaying in console output if not desired
        print(f"Confusion matrix saved to {os.path.join(output_dir, 'log_reg_confusion_matrix_stars.png')}")
    except Exception as e:
        print(f"Could not generate/save confusion matrix plot: {e}")

else:
    print("Skipping TF-IDF + Logistic Regression model training as data is not prepared.")


# --- Model 2: Transformer (DistilBERT) ---

if 'X_train' in locals(): # Check if data split was successful
    print("\n--- Preparing for Model 2: Transformer (DistilBERT) ---")

    # 1. Tokenization
    MODEL_NAME = 'distilbert-base-uncased'
    print(f"Loading tokenizer: {MODEL_NAME}...")
    try:
        tokenizer = DistilBertTokenizer.from_pretrained(MODEL_NAME)
    except Exception as e:
        print(f"Error loading tokenizer: {e}. Cannot proceed with Transformer.")
        tokenizer = None # Set tokenizer to None to prevent further errors


    if tokenizer:
        print("Tokenizing data...")
        start_time = time.time()
        # Convert pandas Series to lists for tokenizer
        train_texts = X_train.tolist()
        test_texts = X_test.tolist()

        # Tokenize - Use appropriate max_length
        MAX_LEN = 256 # Adjust based on review length analysis and memory constraints
        train_encodings = tokenizer(train_texts, truncation=True, padding=True, max_length=MAX_LEN)
        test_encodings = tokenizer(test_texts, truncation=True, padding=True, max_length=MAX_LEN)
        print(f"Tokenization done in {time.time() - start_time:.2f} seconds.")

        # Clear text lists to save memory
        del train_texts, test_texts
        gc.collect()

        # 2. Create PyTorch Datasets
        class SentimentDataset(Dataset):
            def __init__(self, encodings, labels):
                self.encodings = encodings
                self.labels = labels

            def __getitem__(self, idx):
                # Correctly handle accessing dictionary items
                item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
                # Labels should be LongTensor for CrossEntropyLoss
                item['labels'] = torch.tensor(self.labels[idx], dtype=torch.long)
                return item

            def __len__(self):
                # Use 'input_ids' length which is standard
                return len(self.encodings['input_ids'])

        # Convert labels (pandas Series) to numpy arrays BEFORE creating dataset
        Y_train_np = Y_train.values
        Y_test_np = Y_test.values

        train_dataset = SentimentDataset(train_encodings, Y_train_np)
        test_dataset = SentimentDataset(test_encodings, Y_test_np)
        print("PyTorch datasets created.")

        # 3. Define Model
        print(f"Loading pre-trained model: {MODEL_NAME}...")
        try:
            # Load for sequence classification with 3 labels (neg, neu, pos)
            transformer_model = DistilBertForSequenceClassification.from_pretrained(
                MODEL_NAME,
                num_labels=3 # Directly specify the number of output classes
            )
            transformer_model.to(device) # Move model to GPU/CPU
            print("Transformer model loaded and moved to device.")
        except Exception as e:
            print(f"Error loading Transformer model: {e}")
            transformer_model = None # Set model to None

        # 4. Training Setup (only if model loaded successfully)
        if transformer_model:
            print("Setting up training parameters...")
            # Hyperparameters (adjust as needed)
            LEARNING_RATE = 5e-5
            EPOCHS = 1 # Start with 1 epoch for faster iteration, increase later
            BATCH_SIZE = 16 # Adjust based on GPU memory (16 or 32 are common)

            # Dataloaders
            train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
            test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE * 2) # Larger batch size for eval is ok

            # Optimizer and Scheduler
            optimizer = AdamW(transformer_model.parameters(), lr=LEARNING_RATE)
            num_training_steps = EPOCHS * len(train_loader)
            lr_scheduler = get_linear_schedule_with_warmup(
                optimizer,
                num_warmup_steps=0, # You can add warmup steps if desired
                num_training_steps=num_training_steps
            )

            # Loss Function
            loss_fn = torch.nn.CrossEntropyLoss()
            print("Training setup complete.")


            # 5. Training Loop
            print("\n--- Training Transformer Model ---")
            transformer_model.train() # Set model to training mode

            for epoch in range(EPOCHS):
                print(f"\n--- Epoch {epoch + 1}/{EPOCHS} ---")
                epoch_start_time = time.time()
                total_loss = 0

                for i, batch in enumerate(train_loader):
                    optimizer.zero_grad()

                    # Move batch to device
                    input_ids = batch['input_ids'].to(device)
                    attention_mask = batch['attention_mask'].to(device)
                    labels = batch['labels'].to(device)

                    # Forward pass
                    outputs = transformer_model(
                        input_ids=input_ids,
                        attention_mask=attention_mask,
                        labels=labels # Pass labels for loss calculation within the model
                    )

                    loss = outputs.loss # Model calculates loss when labels are provided
                    total_loss += loss.item()

                    # Backward pass and optimization
                    loss.backward()
                    torch.nn.utils.clip_grad_norm_(transformer_model.parameters(), 1.0) # Gradient clipping
                    optimizer.step()
                    lr_scheduler.step()

                    if (i + 1) % 100 == 0: # Print progress every 100 batches
                        print(f"  Batch {i+1}/{len(train_loader)}, Loss: {loss.item():.4f}")

                avg_train_loss = total_loss / len(train_loader)
                epoch_time = time.time() - epoch_start_time
                print(f"Epoch {epoch + 1} completed in {epoch_time:.2f}s. Average Training Loss: {avg_train_loss:.4f}")

            print("Transformer training finished.")

            # Optional: Save the fine-tuned model
            print(f"Saving fine-tuned Transformer model to {output_dir}...")
            try:
                transformer_model.save_pretrained(output_dir)
                tokenizer.save_pretrained(output_dir)
                print("Model and tokenizer saved successfully.")
            except Exception as e:
                 print(f"Error saving model/tokenizer: {e}")


            # 6. Evaluation
            print("\nEvaluating Transformer model...")
            transformer_model.eval() # Set model to evaluation mode
            all_preds_transformer = []
            all_labels_transformer = []

            with torch.no_grad(): # Disable gradient calculations
                for batch in test_loader:
                    input_ids = batch['input_ids'].to(device)
                    attention_mask = batch['attention_mask'].to(device)
                    labels = batch['labels'].to(device)

                    outputs = transformer_model(input_ids=input_ids, attention_mask=attention_mask)
                    logits = outputs.logits
                    predictions = torch.argmax(logits, dim=-1)

                    all_preds_transformer.extend(predictions.cpu().numpy())
                    all_labels_transformer.extend(labels.cpu().numpy())

            print("\nTransformer Classification Report (Stars):")
            print(classification_report(all_labels_transformer, all_preds_transformer, target_names=target_names, digits=4))

            # Optional: Confusion Matrix Plot
            print("Generating Confusion Matrix for Transformer...")
            try:
                cm_transformer = confusion_matrix(all_labels_transformer, all_preds_transformer)
                disp_transformer = ConfusionMatrixDisplay(confusion_matrix=cm_transformer, display_labels=target_names)
                fig, ax = plt.subplots(figsize=(8, 6))
                disp_transformer.plot(cmap=plt.cm.Blues, ax=ax, xticks_rotation='vertical')
                plt.title('Transformer Confusion Matrix (Stars)')
                plt.tight_layout()
                plt.savefig(os.path.join(output_dir, 'transformer_confusion_matrix_stars.png'))
                # plt.show() # Uncomment to display plot directly
                plt.close(fig) # Close the figure
                print(f"Confusion matrix saved to {os.path.join(output_dir, 'transformer_confusion_matrix_stars.png')}")
            except Exception as e:
                print(f"Could not generate/save confusion matrix plot: {e}")

        else:
            print("Skipping Transformer training and evaluation due to model loading error.")
    else:
         print("Skipping Transformer training and evaluation due to tokenizer loading error.")


else:
    print("Skipping Model Training and Evaluation as data is not prepared.")

print("\n--- Script Finished ---")

NLTK data check complete.
Using device: cuda
NLTK data check complete.
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Running in Colab environment.
Model output directory: /content/drive/My Drive/Colab Notebooks/SentimentModels_Stars

Loading review data from: /content/drive/My Drive/Colab Notebooks/SentimentModels_Stars/yelp_academic_dataset_review.json
Review data loaded in 103.98 seconds. Shape: (6990280, 5)

Loading business data from: /content/drive/My Drive/Colab Notebooks/SentimentModels_Stars/yelp_academic_dataset_business.json
Business data loaded in 2.25 seconds. Shape: (150346, 2)

Found 64629 unique target restaurant business IDs.
Reviews filtered by date (2019-01-01 to 2024-12-31). Shape: (2111695, 5)
Reviews filtered by target business IDs. Shape: (1525330, 5)

Final shape for processing: (1525330, 5)
                     review_id  \
194087  F6VdYuJiefNBfn3HNELv0A   
194089  nAMDCKElSKxOhzm

Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`


model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Transformer model loaded and moved to device.
Setting up training parameters...
Training setup complete.

--- Training Transformer Model ---

--- Epoch 1/1 ---
  Batch 100/76266, Loss: 0.5340
  Batch 200/76266, Loss: 0.6773
  Batch 300/76266, Loss: 0.2966
  Batch 400/76266, Loss: 0.4500
  Batch 500/76266, Loss: 0.4007
  Batch 600/76266, Loss: 0.4156
  Batch 700/76266, Loss: 0.4678
  Batch 800/76266, Loss: 0.1845
  Batch 900/76266, Loss: 0.2397
  Batch 1000/76266, Loss: 0.6178
  Batch 1100/76266, Loss: 0.7052
  Batch 1200/76266, Loss: 0.3394
  Batch 1300/76266, Loss: 0.2094
  Batch 1400/76266, Loss: 0.2284
  Batch 1500/76266, Loss: 0.1981
  Batch 1600/76266, Loss: 0.6430
  Batch 1700/76266, Loss: 0.1424
  Batch 1800/76266, Loss: 0.6443
  Batch 1900/76266, Loss: 0.2880
  Batch 2000/76266, Loss: 0.1551
  Batch 2100/76266, Loss: 0.2638
  Batch 2200/76266, Loss: 0.6675
  Batch 2300/76266, Loss: 0.1724
  Batch 2400/76266, Loss: 0.3672
  Batch 2500/76266, Loss: 0.2051
  Batch 2600/76266, Loss

In [ ]:
# Import necessary libraries
import torch
import re
import os
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer

# --- Ensure NLTK data is available (run once if needed) ---
def download_nltk_data():
    """Downloads necessary NLTK data packages."""
    packages = ['punkt', 'stopwords', 'wordnet']
    for package in packages:
        try:
            nltk.data.find(f'corpora/{package}' if package != 'punkt' else f'tokenizers/{package}')
        except LookupError:
            print(f"Downloading NLTK package: {package}...")
            nltk.download(package, quiet=True)
    print("NLTK data check complete.")

download_nltk_data()

# --- Configuration ---
# !! IMPORTANT !! Make sure this matches the directory where you saved
# the star-rating-trained model in the previous script.
try:
    # If running in Colab and saved to Drive
    from google.colab import drive
    # Ensure Drive is mounted if needed (might not be necessary if already mounted)
    try:
        drive.mount('/content/drive', force_remount=True)
    except: # Handle cases where drive might already be mounted or not in Colab
        pass
    model_save_directory = '/content/drive/My Drive/Colab Notebooks/SentimentModels_Stars'
    print("Assuming Colab environment for model path.")
except ModuleNotFoundError:
    # If running locally
    model_save_directory = './SentimentModels_Stars'
    print("Assuming local environment for model path.")

print(f"Attempting to load model from: {model_save_directory}")

# Check for GPU
device = torch.device('cuda') if torch.cuda.is_available() else torch.device('cpu')
print(f"Using device: {device}")

# Define the mapping from integer labels (used during training) to sentiment strings
# Must match the mapping used when training the star-based model:
# 0: Negative (1-2 Stars), 1: Neutral (3 Stars), 2: Positive (4-5 Stars)
label_map = {
    0: 'Negative',
    1: 'Neutral',
    2: 'Positive'
}

# --- Text Preprocessing Function (Same as before) ---
lemmatizer = WordNetLemmatizer()
stop_words_set = set(stopwords.words('english'))

def preprocess_text(text):
    """Cleans and preprocesses text data."""
    if not isinstance(text, str):
        return ""
    text = text.lower() # Lowercase
    text = re.sub(r'http\S+|www\S+|https\S+', '', text, flags=re.MULTILINE) # Remove URLs
    text = re.sub(r'\@\w+|\#', '', text) # Remove mentions and hashtags
    text = re.sub(r'[^\w\s]', '', text) # Remove punctuation
    text = re.sub(r'\d+', '', text) # Remove numbers
    tokens = word_tokenize(text) # Tokenize
    processed_tokens = [
        lemmatizer.lemmatize(word) for word in tokens
        if word not in stop_words_set and word.isalpha() and len(word) > 1
    ]
    return ' '.join(processed_tokens)

# --- Load Model and Tokenizer ---
try:
    print("Loading fine-tuned tokenizer...")
    tokenizer = AutoTokenizer.from_pretrained(model_save_directory)
    print("Loading fine-tuned model...")
    model = AutoModelForSequenceClassification.from_pretrained(model_save_directory)
    model.to(device) # Move model to the appropriate device
    model.eval() # Set model to evaluation mode
    print("Model and tokenizer loaded successfully.")
    model_loaded = True
except OSError as e:
    print(f"Error loading model/tokenizer from {model_save_directory}: {e}")
    print("Please ensure the directory exists and contains the saved model files ('pytorch_model.bin', 'config.json', 'tokenizer_config.json', etc.).")
    print("Cannot proceed with predictions.")
    model_loaded = False
except Exception as e:
    print(f"An unexpected error occurred during loading: {e}")
    model_loaded = False

# --- Prediction Function ---
def predict_overall_sentiment(text, loaded_model, loaded_tokenizer):
    """Predicts overall sentiment for a given text using the star-rating model."""
    if not model_loaded:
         return "Error: Model not loaded."

    # 1. Preprocess the input text
    processed_text = preprocess_text(text)
    if not processed_text:
        return "Input text is empty after preprocessing."

    # 2. Tokenize
    # Adjust max_length if needed, should match training if possible
    inputs = loaded_tokenizer(
        processed_text,
        return_tensors='pt',
        truncation=True,
        padding=True,
        max_length=256 # Use the same MAX_LEN as during training if possible
    )

    # 3. Move inputs to the same device as the model
    inputs = {k: v.to(device) for k, v in inputs.items()}

    # 4. Predict
    with torch.no_grad(): # Disable gradient calculations for inference
        outputs = loaded_model(**inputs)
        logits = outputs.logits

    # 5. Get prediction index
    prediction_index = torch.argmax(logits, dim=-1).squeeze().item()

    # 6. Map index to sentiment label
    sentiment = label_map.get(prediction_index, "Unknown Label")

    return sentiment

# --- User Input Loop ---
if model_loaded:
    print("\n--- Overall Sentiment Predictor ---")
    print("Enter your review text below. Type 'quit' to exit.")

    while True:
        user_input = input("\nEnter review text: ")
        if user_input.lower() == 'quit':
            break
        if not user_input.strip():
            print("Please enter some text.")
            continue

        # Predict sentiment
        predicted_sentiment = predict_overall_sentiment(user_input, model, tokenizer)

        print(f"Predicted Sentiment: {predicted_sentiment}")

else:
    print("\nSkipping prediction loop as the model could not be loaded.")

print("\n--- Exiting Sentiment Predictor ---")

NLTK data check complete.
Mounted at /content/drive
Assuming Colab environment for model path.
Attempting to load model from: /content/drive/My Drive/Colab Notebooks/SentimentModels_Stars
Using device: cuda
Loading fine-tuned tokenizer...
Loading fine-tuned model...
Model and tokenizer loaded successfully.

--- Overall Sentiment Predictor ---
Enter your review text below. Type 'quit' to exit.
Predicted Sentiment: Positive
Predicted Sentiment: Negative
Predicted Sentiment: Neutral
